In [ ]:
Ctimport pandas as pd; import numpy as np; from sklearn.model_selection import train_test_split; from sklearn.metrics import mean_squared_error as mse; from sklearn.metrics.pairwise import cosine_similarity;from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse, r2_score

In [ ]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip /content/ml-latest-small.zip
!mv /content/ml-latest-small /content/data
!rm /content/ml-latest-small.zip

--2026-02-21 06:31:34--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  4.32MB/s    in 0.2s    

2026-02-21 06:31:35 (4.32 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  /content/ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


#1. Подготовка данных

In [ ]:
d = pd.read_csv('data/ratings.csv').drop('timestamp', axis=1); d = d.dropna(); display(d.head(1),d.describe())
u_f = d.groupby('userId' )['rating'].count(); d = d[d.userId .isin(u_f[u_f > 50].index)]
m_f = d.groupby('movieId')['rating'].count(); d = d[d.movieId.isin(m_f[m_f > 20].index)]


tr, te = train_test_split(d, test_size=0.2, random_state=42)
mx = tr.pivot(index='userId', columns='movieId', values='rating')
#0 - не рейтинг в большинстве случаев, поэтому вычтем среднее, чтобы увидеть именно разницу в оценке для каждого пользователя, нежели сравнивать разные шкалы оценок от разных пользователей
user_means = mx.mean(axis=1).values.reshape(-1, 1)
#Для movie использую то же среднее по человеку, потому что шкалы оценок для всех юзеров разные, и будет разумнее их нормализовать
mx = mx.sub(mx.mean(axis=1), axis=0).fillna(0)
u_idx = mx.index; i_idx = mx.columns; mx_v = mx.values

,userId,movieId,rating
0,1,1,4.0


,userId,movieId,rating
count,100836.000000,100836.000000,100836.000000
mean,326.127564,19435.295718,3.501557
std,182.618491,35530.987199,1.042529
min,1.000000,1.000000,0.500000
25%,177.000000,1199.000000,3.000000
50%,325.000000,2991.000000,3.500000
75%,477.000000,8122.000000,4.000000
max,610.000000,193609.000000,5.000000


In [ ]:
print(mx.iloc[:5,:5])

movieId         1         2         3    5         6
userId                                              
1       -0.421769  0.000000 -0.421769  0.0 -0.421769
4        0.000000  0.000000  0.000000  0.0  0.000000
6        0.000000  0.256198  1.256198  0.0  0.256198
7        0.000000  0.000000  0.000000  0.0  0.000000
10       0.000000  0.000000  0.000000  0.0  0.000000


#2. Коллаборативная фильтрация

##2.1 User-Based

In [ ]:
u_s = cosine_similarity(mx_v)
def pred_u(s, m, means, filter_anti_neighbours=False):
  if filter_anti_neighbours:s=np.maximum(s,0)
  return means + s @ m / np.array([np.abs(s).sum(axis=1)]).T
u_p = pred_u(u_s, mx_v, user_means)
print(u_p[:5])

[[4.40212106 4.35132338 4.37261725 ... 4.41805863 4.42633987 4.43371843]
 [3.58950171 3.59065972 3.57081609 ... 3.58716941 3.59932566 3.58924551]
 [3.81254513 3.80147842 3.808066   ... 3.75073915 3.7417102  3.75973658]
 [3.43310789 3.3181491  3.32031546 ... 3.36536451 3.35983595 3.36429742]
 [3.29717268 3.37562173 3.33998571 ... 3.32154683 3.32011036 3.32383061]]


##2.2 Item-Based

In [ ]:
i_s = cosine_similarity(mx_v.T)
def pred_i(m,s, means, filter_anti_neighbours=False):
  if filter_anti_neighbours:s=np.maximum(s,0)
  return means + m @ s / np.array([np.abs(s).sum(axis=1)])
i_p = pred_i(mx_v, i_s, user_means)
print(i_p[:5])

[[4.42159173 4.38970834 4.39465864 ... 4.4183345  4.43644355 4.43421937]
 [3.60040572 3.5753915  3.57543785 ... 3.58047779 3.61552722 3.58746171]
 [3.76471507 3.75971604 3.78165963 ... 3.75107696 3.74243995 3.76010348]
 [3.39197302 3.32895815 3.32959582 ... 3.37376349 3.36644769 3.37105451]
 [3.31552333 3.34191851 3.33156992 ... 3.31950843 3.3085505  3.3182854 ]]


#3. Рекоммендации

In [ ]:
def get_recommendations(user_id, pred_matrix, original_mx, item_ids, n=5):
    user_preds = pred_matrix.loc[user_id]#строка предсказаний для юзера
    #фильмы, которые юзер уже оценил, рекоммендовать не надо.
    already_rated = original_mx.loc[user_id]
    already_rated = already_rated[already_rated > 0].index
    recommendations = user_preds.drop(already_rated)

    return recommendations.sort_values(ascending=False).head(n)


u_p_df = pd.DataFrame(u_p, index=u_idx, columns=i_idx)
i_p_df = pd.DataFrame(i_p, index=u_idx, columns=i_idx)
print(get_recommendations(1, u_p_df, mx, i_idx))
print(get_recommendations(1, i_p_df, mx, i_idx))

movieId
318     4.631069
858     4.596857
4993    4.572780
7153    4.567646
7361    4.560437
Name: 1, dtype: float64
movieId
778     4.474083
1285    4.472502
2300    4.470571
2968    4.468029
1345    4.467295
Name: 1, dtype: float64


#4. Оценка

In [ ]:
def evaluate_model(preds_matrix, test_df, u_idx, i_idx):
    y_true = []
    y_pred = []
    pd_preds = pd.DataFrame(preds_matrix, index=u_idx, columns=i_idx)

    for _, row in test_df.iterrows():
        u, m, r = int(row.userId), int(row.movieId), row.rating
        if u in u_idx and m in i_idx:
            p = pd_preds.loc[u, m]
            if p > 0:#считаем только там где прогноз есть.
                y_true.append(r)
                y_pred.append(p)

    return {
        "MAE": mae(y_true, y_pred),
        "MSE": mse(y_true, y_pred),
        "RMSE": np.sqrt(mse(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }


u_p_raw  = pred_u(u_s, mx_v, user_means, filter_anti_neighbours=False)
u_p_filt = pred_u(u_s, mx_v, user_means, filter_anti_neighbours=True)

i_p_raw  = pred_i(mx_v, i_s, user_means, filter_anti_neighbours=False)
i_p_filt = pred_i(mx_v, i_s, user_means, filter_anti_neighbours=True)

results = pd.DataFrame([
    evaluate_model(u_p_raw, te, u_idx, i_idx),
    evaluate_model(u_p_filt, te, u_idx, i_idx),
    evaluate_model(i_p_raw, te, u_idx, i_idx),
    evaluate_model(i_p_filt, te, u_idx, i_idx)
], index=['User-Raw', 'User-Filtered', 'Item-Raw', 'Item-Filtered'])

display(results.round(4))

,MAE,MSE,RMSE,R2
User-Raw,0.6746,0.7822,0.8844,0.2183
User-Filtered,0.6702,0.7746,0.8801,0.2258
Item-Raw,0.6792,0.7893,0.8884,0.2112
Item-Filtered,0.6768,0.7847,0.8859,0.2158
